#1. Introduction to Spark Architecture
#####Apache Spark is a distributed computing engine designed to process large-scale data by distributing it across a cluster of machines.

### The Cluster Components

Spark operates on a **Master-Slave architecture** consisting of the following entities:

* **Cluster Manager:** Manages resources and the allocation of nodes.
* **Driver Node (Master):** The central controller. It orchestrates tasks, breaks down code into transformations, and communicates with the cluster manager to request worker nodes.
* **Worker Nodes (Slaves):** The machines that actually execute the tasks and process the data.

### The Execution Hierarchy

When code is submitted, Spark organises the execution into a specific hierarchy:

1. **Job:** Triggered by an action; typically corresponds to the code in a single notebook cell.
2. **Stage:** A job is divided into stages based on data shuffling requirements.
3. **Task:** The smallest unit of work, executed on a single partition of data within a worker node.

### Core Benefits

* **In-Memory Computation:** Unlike Hadoop, which uses disk memory, Spark processes data in RAM, significantly increasing speed.
* **Fault Tolerance:** The ability to trace back transformations to recover lost data partitions.
* **Lazy Evaluation:** Spark does not execute transformations immediately; instead, it builds a **Logical Plan**.

---

## 2\. Lazy Evaluation and Execution

Spark distinguishes between **Transformations** (e.g., `select`, `filter`) and **Actions** (e.g., `show`, `display`, `collect`).

* **Mechanism:** When you define transformations, Spark stores them in a Directed Acyclic Graph (DAG) or logical plan. It optimises this plan (e.g., reordering filters for efficiency).
* **Trigger:** Execution only occurs when an **Action** is triggered.

| **Interview Tip** | **Question:** Why is lazy evaluation beneficial? **Answer:** It allows Spark to optimise the execution plan as a whole rather than executing inefficient, step-by-step commands. |
| ----------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |


###Also we dont need to use subset keyword

# Reading CSV Data with Spark

This notebook demonstrates how to read a CSV file from a Unity Catalog volume using PySpark.

In [0]:
# List files in a Unity Catalog volume
# dbutils.fs.ls requires the FULL path: /Volumes/{catalog}/{schema}/{volume}
volume_path = "/Volumes/workspace/default/pyspark_files"

files = dbutils.fs.ls(volume_path)

for f in files:
    print(f"{f.name}  ({f.size / 1024:.1f} KB)")

In [0]:
path_volume = "/Volumes/workspace/default/pyspark_files"
file_name = "BigMart Sales.csv"

df = spark.read.csv(f"{path_volume}/{file_name}",
  header=True,
  inferSchema=True,
  sep=",")

display(df)

In [0]:
df2 = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/default/pyspark_files/BigMart Sales.csv"))

df2.show()

display(df)  → Databricks interactive table


df.show()    → PySpark console-style output

# Reading JSON Data with Spark


In [0]:
df3 = spark.read.json("/Volumes/workspace/default/pyspark_files/drivers.json", multiLine=False)
display(df3)

# Schema Defination

In [0]:
df.printSchema()

### 1. DDL schema (to change schema)

In [0]:
ddl_schema = '''
                Item_Identifier STRING, 
                Item_Weight STRING, 
                Item_Fat_Content STRING,
                Item_Visibility DOUBLE,
                Item_Type STRING,
                Item_MRP DOUBLE,
                Outlet_Identifier STRING,
                Outlet_Establishment_Year INT, 
                Outlet_Size STRING, 
                Outlet_Location_Type STRING,
                Outlet_Type STRING, 
                Item_Outlet_Sales DOUBLE    
            '''
#Item_Weight: DOUBLE -> STRING

In [0]:
df = spark.read.format('csv').schema(ddl_schema)\
    .option('header', True)\
    .load('/Volumes/workspace/default/pyspark_files/BigMart Sales.csv')

df.printSchema()

### 2. Struct_Type

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
schema = StructType([
    StructField("Item_Identifier", StringType(), True),
    StructField("Item_Weight", StringType(), True),
    StructField("Item_Fat_Content", StringType(), True),
    StructField("Item_Visibility", StringType(), True),
    StructField("Item_Type", StringType(), True),
    StructField("Item_MRP", StringType(), True),
    StructField("Outlet_Identifier", StringType(), True),
    StructField("Outlet_Establishment_Year", StringType(), True),
    StructField("Outlet_Size", StringType(), True),
    StructField("Outlet_Location_Type", StringType(), True),
    StructField("Outlet_Type", StringType(), True),
    StructField("Item_Outlet_Sales", StringType(), True)
])

df = spark.read.schema(schema).csv('/Volumes/workspace/default/pyspark_files/BigMart Sales.csv', header=True)

In [0]:
display(df)